In [ ]:
import os
import re
import pathlib

import matplotlib.patches as mpatches
from matplotlib.transforms import blended_transform_factory
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import OrderedDict
from matplotlib.lines import Line2D
from IPython.display import HTML, display

In [ ]:
# --- 1. Φόρτωση Αρχείου (Υποθέτουμε ότι η διαδρομή είναι σωστή) ---
csv_name = r"a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\12_20250515_globalDoughnutData_2000-2022.csv"

# 1) σωστός έλεγχος αρχείου
if os.path.exists(csv_name):
    file_path = csv_name
elif os.path.exists(os.path.join('myData', csv_name)):
    file_path = os.path.join('myData', csv_name)
else:
    raise FileNotFoundError(f"❌ Το αρχείο {csv_name} δεν βρέθηκε! Βεβαιώσου ότι είναι στον ίδιο φάκελο με το Notebook.")

df_ts = pd.read_csv(file_path)

# ΚΑΘΑΡΙΣΜΟΣ
df_ts.columns = df_ts.columns.str.strip()
year_col = [col for col in df_ts.columns if 'date' in col.lower()][0]

# --- 2. Λεξικό για Πλήρεις Περιγραφές (Metadata) ---
names_map = {
    'CO2': ('Connectivity', 'Population not accessing the internet'),
    'CO1': ('Connectivity', 'Urban population lacking convenient access to public\ntransport'),
    'ED1': ('Education', 'Adult population (aged 15+ years) who are illiterate'),
    'ED2': ('Education', 'Young adult population (aged 21-23 years) with\nincomplete upper secondary education'),
    'EN1': ('Energy', 'Population lacking access to electricity'),
    'EN2': ('Energy', 'Population lacking access to clean fuels and\ntechnologies for cooking, heating and lighting'),
    'EQ1': ('Equality', 'Population-weighted score on the Gender Inequality\nIndex (global gap between women and men in terms of\nreproductive health, empowerment and employment)'), 
    'NU2': ('Food', 'Population with moderate to severe food insecurity'),
    'NU1': ('Food', 'Population undernourished'),
    'HE2': ('Health', 'Population living in countries without high coverage\nof essential health services (Universal Health\nCoverage Index score less than 60 out of 100)'), 
    'HE1': ('Health', 'Population living in countries with under-5\nmortality rate exceeding 25 per 1,000 live births'), 
    'HO1': ('Housing', 'Urban population living in slums or informal settlements'),
    'IW1': ('Income and work', 'Population living below the societal poverty line,\n set at half their country\'s median household\nincome or at least $15 a day'), 
    'IW2': ('Income and work', 'Population of young people (aged 15-24 years) not\nin employment, education or training'),
    'PJ1': ('Peace and justice', 'Population stating that they perceive\nwidespread corruption in government and business'),
    'PJ2': ('Peace and justice', 'Population living in countries with a homicide rate\nof 5 or more per 100,000'),
    'PV1': ('Political voice', 'Population living in countries governed by an autocratic regime'),
    'SC2': ('Social cohesion', 'Population living in countries with a Palma ratio of 2 or more\n(the income share of the richest 10% of people relative\nto the poorest 40%)'),
    'SC1': ('Social cohesion', 'Population stating that they are without someone to\ncount on in times of trouble'),
    'WA1': ('Water', 'Population lacking access to safely managed drinking water'),
    'WA2': ('Water', 'Population lacking access to safely managed sanitation'),
}

# --- 3. Δυναμικός Υπολογισμός Πίνακα (Table 1) ---
table_rows = []

df_social = df_ts[df_ts['domain'] == 'social']

for code, (dim_name, ind_name) in names_map.items():
    
    # 1. Φιλτράρουμε & Ταξινομούμε όλα τα δεδομένα (συμπεριλαμβανομένων των NaN)
    subset = df_social[df_social['indCode'] == code].sort_values(by=year_col).reset_index(drop=True)
    
    if not subset.empty and code != 'EQ2': 
        
        # 2. Εντοπίζουμε την πρώτη ΓΡΑΜΜΗ ΧΩΡΙΣ NaN (First Non-Missing Row)
        # Χρησιμοποιούμε τη μέθοδο .first_valid_index() αφού βάλουμε το έτος ως index.
        # Ωστόσο, για να διατηρήσουμε τη σειρά, ας φιλτράρουμε με Pandas boolean mask.
        
        # --- ΕΥΡΕΣΗ ΕΓΚΥΡΗΣ ΑΡΧΗΣ ---
        # 2) χρήση τελευταίας μη-NaN τιμής για το τέλος
        valid_subset = subset.dropna(subset=['value']).reset_index(drop=True)
        if not valid_subset.empty:
            min_year = valid_subset[year_col].iloc[0]
            val_start = valid_subset['value'].iloc[0]
            max_year = valid_subset[year_col].iloc[-1]   # ή valid_subset[year_col].iloc[-1] αν θες το έτος της τελευταίας μη-NaN
            val_end = valid_subset['value'].iloc[-1]
            
            table_rows.append({
                'dimension': dim_name,
                'indicator': ind_name,
                'date_first': int(min_year),
                'date_last': int(max_year),
                'value_first': val_start,
                'value_last': val_end
            })

# Δημιουργία DataFrame
df_table1 = pd.DataFrame(table_rows)
mask = df_table1['indicator'].astype(str).str.contains(
    'Urban population lacking convenient access to public', na=False
)
if mask.any():
    df_table1.loc[mask, 'date_last'] = 2020

# Προετοιμασία για εμφάνιση
column_order = [
    'dimension', 'indicator',
    'date_first', 'date_last',
    'value_first', 'value_last'
]

df_table1 = df_table1[column_order]

# Δημιουργία MultiIndex headers: πάνω γραμμή (date/value με colspan=2), κάτω γραμμή (first/last)

cols_multi = pd.MultiIndex.from_tuples([
    ('dimension',''),
    ('indicator',''),
    ('date','first'),
    ('date','last'),
    ('value','first'),
    ('value','last'),
])
df_display = df_table1.copy()
df_display.columns = cols_multi

print("\n### Table 1: The Social Foundation (ΤΕΛΙΚΗ ΕΚΔΟΣΗ)")



css = """
<style>
table.dataframe { border-collapse: collapse; width: 100%; }
table.dataframe thead th {
  text-align: center;            /* center header rows */
  padding: 6px;
  border: 1px solid #ccc;
}
table.dataframe tbody td {
  text-align: left;              /* left-align body rows */
  padding: 6px;
  border: 1px solid #ccc;
}
</style>
"""

html = css + df_display.to_html(index=False, float_format="%.2f", na_rep="-")
display(HTML(html))

In [ ]:
# --- 1. Επέκταση του Λεξικού (Εικολογικοί Δείκτες) ---

# Το λεξικό names_map που έχεις ήδη στο notebook σου, τώρα πρέπει να το επεκτείνουμε.
# Αν έχεις ακόμα ανοιχτό το κελί του Table 1, μπορείς να τρέξεις μόνο αυτό:

ecological_indicators = {
    'CC1': ('Climate change', 'Atmospheric carbon dioxide concentration, parts per million\n(at most 350 ppm CO2)'),
    'CC2': ('Climate change', 'Human-induced radiative forcing at the top of the atmosphere,\n Watt per square metre (at most 1 W m**(-2))'),
    'OA1': ('Ocean acidification', 'Average saturation state of aragonite at the ocean surface\n(at least 80% of pre-industrial saturation state of 3.44 Ωarag)'),
    'OD1': ('Ozone depletion', 'Concentration of ozone in the stratosphere, Dobson units\n(at most 5% decrease with respect to 1964-1980 value of 290 DU)'),
    'CP1': ('Chemical pollution', 'Production of hazardous chemicals, millions of tonnes\nper year (at most 5% of the 1,200 Mt of total chemicals\nproduced in year 2000)'),
    'NP2': ('Nutrient pollution', 'Nitrogen applied to land as fertilizer, millions of tonnes per year\n(at most 62 Mt per year)'),
    'NP1': ('Nutrient pollution', 'Phosphorus applied to land as fertilizer, millions of tonnes per year\n(at most 6.2 Mt per year)'),
    'AP1': ('Air pollution', "Arithmetic Error Asymmetry between Earth's hemispheres of sunlight\nreaching the surface, owing to differences in atmospheric particle\nconcentration (at most 0.1 inter-hemispheric difference in Aerosol\nOptical Depth)"),
    'FD1': ('Freshwater disruption', 'Proportion of land area with human-induced disturbance of blue-water\nflow deviating from Holocene variability (at most 10.2%)'),
    'FD2': ('Freshwater disruption', 'Proportion of land area with root-zone soil moisture deviating from\nHolocene variability (at most 11.1%)'),
    'LC1': ('Land conversion', 'Area of forested land as a proportion of forest-covered land before\nhuman alteration (at least 75% of 64 million square kilometres)'),
    'BB1': ('Biodiversity breakdown', 'Rate of species extinctions per million species\nyears (at most 10 E/MSY)'),
    'BB2': ('Biodiversity breakdown', 'Human appropriation of net primary productivity,\nbillions of tonnes of carbon per year (at most 10% of 55.9 Gt C)')
}

# Συγχωνεύουμε τα λεξικά (Social + Ecological)
names_map_full = {**names_map, **ecological_indicators}

print("✅ Το λεξικό names_map_full δημιουργήθηκε.")

In [ ]:

if 'df_data' not in globals():
    p = r"a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\2_20250108_doughnutv3-data.csv"
    if os.path.exists(p):
        df_data = pd.read_csv(p)
        df_data.columns = df_data.columns.str.strip()
    else:
        raise FileNotFoundError(f"Το αρχείο df_data δεν βρέθηκε στη διαδρομή: {p}")

if 'df_ts' not in globals():
    raise NameError("df_ts is not defined — τρέξε πρώτα το κελί που φορτώνει το globalDoughnutData CSV")

if 'df_merged' not in globals():
    df_merged = pd.merge(
        df_data,
        df_ts[['indicator', 'indCode', 'domain']].drop_duplicates(),
        on='indicator',
        how='left',
        suffixes=('_data', '_ts')
    )
    df_merged.columns = df_merged.columns.str.strip()

# Build table rows preferring rows found in df_merged (includes 'chemical' matches)
table_rows_eco = []
year_candidates = [c for c in df_merged.columns if c.lower() in ('year_data','date_data','year','date')]
year_col = year_candidates[0] if year_candidates else 'date'

for code, (dim_name, ind_name) in ecological_indicators.items():
    subset = df_merged[df_merged.get('indCode','').astype(str) == code].copy()

    if subset.empty:
        try:
            name_snippet = ind_name.split(',')[0].split('(')[0].strip().lower()
            ind_col = df_merged['indicator'].astype(str).str.lower()
            mask = ind_col.str.contains(name_snippet, na=False) | ind_col.str.contains(code.lower(), na=False)
            if 'chem' in ind_name.lower() or 'chemical' in ind_name.lower() or code.startswith('CP'):
                mask = mask | ind_col.str.contains('chemical', na=False) | ind_col.str.contains('chem', na=False)
            subset = df_merged[mask].copy()
        except Exception:
            pass

    if not subset.empty and 'domain_data' in subset.columns:
        eco_rows = subset[subset['domain_data'] == 'ecological']
        if not eco_rows.empty:
            subset = eco_rows

    if subset.empty and 'df_ts' in globals():
        alt3 = df_ts[df_ts.get('indCode','').astype(str) == code].copy()
        if not alt3.empty:
            subset = alt3

    if subset.empty:
        continue

    yc_candidates = [c for c in subset.columns if c.lower() in ('year_data','date_data','year','date')]
    yc = yc_candidates[0] if yc_candidates else year_col

    subset = subset.sort_values(by=yc).reset_index(drop=True)
    valid_subset = subset.dropna(subset=['value']).reset_index(drop=True)
    if valid_subset.empty:
        continue

    min_year = int(valid_subset[yc].iloc[0])
    val_start = valid_subset['value'].iloc[0]
    max_year = int(valid_subset[yc].iloc[-1])
    val_end = valid_subset['value'].iloc[-1]

    table_rows_eco.append({
        'dimension': dim_name,
        'indicator': ind_name,
        'date_first': min_year,
        'date_last': max_year,
        'value_first': val_start,
        'value_last': val_end
    })

df_table2 = pd.DataFrame(table_rows_eco)

# Prepare display: alphabetical by indicator, no index, MultiIndex headers, CSS like Table 1
display_df = df_table2[['dimension','indicator','date_first','date_last','value_first','value_last']].sort_values(by='dimension').reset_index(drop=True)
display_df.columns = pd.MultiIndex.from_tuples([
    ('dimension',''), ('indicator',''),
    ('date','first'), ('date','last'),
    ('value','first'), ('value','last'),
])
display_df.columns.set_names([None, None], inplace=True)


css = """
<style>
table.dataframe { border-collapse: collapse; width: 100%; font-family: Arial, Helvetica, sans-serif; }
table.dataframe thead th {
  text-align: center;            /* center header rows */
  padding: 8px;
  border: 1px solid #ccc;
  vertical-align: middle;
}
table.dataframe tbody td {
  text-align: left;              /* left-align body rows */
  padding: 8px;
  border: 1px solid #ccc;
}
</style>
"""
html = css + display_df.to_html(index=False, float_format="%.2f", na_rep="-")
display(HTML(html))

In [ ]:
# load df_grp if missing

p = r"a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\11_20250127-grpDataAndShares.csv"
if 'df_grp' not in globals():
    if not os.path.exists(p):
        raise FileNotFoundError(f"Required file not found: {p}")
    df_grp = pd.read_csv(p)
    df_grp.columns = df_grp.columns.str.strip()

# ensure date numeric and pick target year
df_grp['date'] = pd.to_numeric(df_grp.get('date'), errors='coerce')
if df_grp['date'].notna().any():
    TARGET_YEAR = int(df_grp['date'].max())
else:
    raise RuntimeError("No numeric 'date' values in df_grp — διορθώστε το CSV ή ορίστε TARGET_YEAR χειροκίνητα")

# Φιλτράρισμα βάσει domain
DOMAIN_TO_PLOT = 'ecological'   # άλλαξε σε 'social' αν θες το social plot

# φτιάχνουμε domain_df με μόνο τις γραμμές του domain
domain_df = df_grp.loc[df_grp['domain'].astype(str) == DOMAIN_TO_PLOT].copy()
if domain_df.empty:
    raise RuntimeError(f"No rows with domain == '{DOMAIN_TO_PLOT}' found in df_grp")

# groups μόνο από το domain_df (ασφαλές: δεν θα έχει ομάδες που δεν εμφανίζονται σε αυτό το domain)
groups_present = list(domain_df['group'].astype(str).unique())

preferred = ['bottom40','middle40','top20','bottom-40','middle-40','top-20']
order_for_plot = [g for g in preferred if g in groups_present] + [g for g in groups_present if g not in preferred]

color_map = {
    'bottom40':"#ffd700",'bottom-40':'#ffd700',
    'middle40':"#d8bfd8",'middle-40':'#d8bfd8',
}
def _norm(s): return str(s).lower().replace('-','').replace('_','').replace(' ','')
canonical_colors = { _norm(k): v for k,v in color_map.items() }

# palette μόνο για τις ομάδες που υπάρχουν στο domain_df
palette_dict = { g: canonical_colors.get(_norm(g), '#777777') for g in groups_present }

# και μετά χτίζεις το df που θα περάσεις στο plotting από domain_df
# π.χ. για year filter:
df_plot = domain_df.loc[(domain_df['date'] == TARGET_YEAR)].copy()
if df_plot.empty:
    raise RuntimeError(f"No {DOMAIN_TO_PLOT} rows for TARGET_YEAR in df_grp")


# Συνάρτηση κανονικοποίησης ονομάτων ομάδων
def _norm(s):
    return str(s).lower().replace('-', '').replace('_', '').replace(' ', '')

# build canonical lookup from color_map using normalized keys
canonical_colors = { _norm(k): v for k, v in color_map.items() }

# compose palette dict with exactly the group labels present in the dataframe
palette_dict = {}
default_color = '#4682b4'
for g in groups_present:
    palette_dict[g] = canonical_colors.get(_norm(g), default_color)

# build df for ecological indicators at TARGET_YEAR
# Βεβαιώσου ότι έχεις ορίσει DOMAIN_TO_PLOT = 'ecological' και domain_df νωρίτερα στο κελί.
df_ec_plot = domain_df.loc[domain_df['date'] == TARGET_YEAR].copy()

if 'ratio' not in df_ec_plot.columns:
    raise RuntimeError("'ratio' column missing in df_ec_plot")
df_ec_plot['percent'] = pd.to_numeric(df_ec_plot['ratio'], errors='coerce') * 100

# sorted unique indicators (case-insensitive)
indicators = sorted(df_ec_plot['indicator'].astype(str).unique(), key=lambda s: s.lower())

# single catplot (trellis)
g = sns.catplot(
    data=df_ec_plot,
    x='group',
    y='percent',
    col='indicator',
    col_order=indicators,
    kind='bar',
    col_wrap=3,
    hue='group',
    hue_order=order_for_plot,
    palette=palette_dict,
    order=order_for_plot,
    height=3.0,        # λίγο μικρότερα panels
    aspect=0.95,
    sharey=True,
    dodge=False,
    legend=False,
)

g.set_axis_labels("", "")

# translation map
translate_map = {
    'h2o': 'Blue water footprint',
    'water': 'Blue water footprint',
    'co2': 'Carbon footprint',
    'carbon': 'Carbon footprint',
    'hanpp': 'HANPP footprint',
    'human appropriation': 'HANPP footprint',
    'npp': 'HANPP footprint',
    'nitrogen': 'Nitrogen footprint',
    'n': 'Nitrogen footprint',
    'phosphorus': 'Phosphorus footprint',
    'p': 'Phosphorus footprint',
    'species': 'Species-loss footprint',
    'biodiversity': 'Species-loss footprint',
}

description_map = { code: desc for code, (dim, desc) in names_map_full.items() }

def translate_indicator(name):
    if not isinstance(name, str):
        return name
    key_orig = name.strip()
    # try exact indCode lookup (CO2, NU1, κλπ)
    desc = description_map.get(key_orig.upper())
    if desc:
        return desc
    # fallback: προηγούμενη λογική με translate_map
    key = key_orig.lower().replace('_', ' ')
    tokens = re.findall(r'[a-z]+', key)
    for t in tokens:
        if t in translate_map:
            return translate_map[t]
    for k in sorted(translate_map.keys(), key=lambda x: -len(x)):
        if k in key:
            return translate_map[k]
    return key_orig.replace('_', ' ').title()

# labels mapping for appearance
group_label_map = {
    _norm('bottom40'): 'Bottom-40',
    _norm('bottom-40'): 'Bottom-40',
    _norm('middle40'): 'Middle-40',
    _norm('middle-40'): 'Middle-40',
    _norm('top20'): 'Top-20',
    _norm('top-20'): 'Top-20',
}

# plotting scales / ticks
fixed_ticks = [0, 200, 400, 600, 800, 1000]
y_max = max(1000, df_ec_plot['percent'].max() * 1.05)
if y_max < fixed_ticks[-1]:
    y_max = fixed_ticks[-1]

col_wrap = 3
n_groups = len(order_for_plot)
# baseline positions; θα τα μετατοπίσουμε λίγο δεξιά (+0.03) για καλύτερο κέντρο
xs_norm = np.linspace(0.12, 0.88, n_groups) if n_groups > 0 else []
xs_norm = xs_norm + 0.03  # small right shift to center the labels over bars

# style facets: smaller titles, translate indicator names, angled top labels (45°), smaller fontsize
for i, ax in enumerate(g.axes.flatten()):
    ax.set_ylim(0, y_max)
    ax.set_yticks(fixed_ticks)
    ax.set_yticklabels([f"{t}%" for t in fixed_ticks], fontsize=8)

    ind_raw = ax.get_title().split('=')[-1].strip()
    ax.set_title(translate_indicator(ind_raw), fontsize=10, y=0.98)
    ax.set_xlabel('')
    if i % col_wrap == 0:
        ax.set_ylabel("Ecological overshoot", fontsize=10)
    else:
        ax.set_ylabel("") # Ensure only the leftmost axis has the label
    # top row: draw the group labels above panels, angled 45°, centered and slightly shifted right
    if i < col_wrap and n_groups > 0:
        for j, grp in enumerate(order_for_plot):
            normg = _norm(grp)
            grp_label = group_label_map.get(normg, str(grp).replace('-', ' ').replace('_', ' ').title())
            ax.text(xs_norm[j], 1.06, grp_label,
                    transform=ax.transAxes,
                    ha='center', va='bottom',     # center horizontally over each bar group
                    rotation=45, fontsize=9, fontstyle='italic')
        ax.tick_params(axis='x', rotation=0, labelsize=8)
    else:
        # lower rows: hide x ticklabels entirely
        ax.set_xticklabels([])
        ax.tick_params(labelbottom=False)

# λίγο περισσότερο χώρος πάνω για να μην επικαλύπτονται οι τίτλοι
g.figure.subplots_adjust(top=0.88)
plt.show()

In [ ]:
p = r"a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\11_20250127-grpDataAndShares.csv"
if 'df_grp' not in globals():
    # Load df_grp if missing
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    df_grp = pd.read_csv(p)
    df_grp.columns = df_grp.columns.str.strip()
    df_grp['date'] = pd.to_numeric(df_grp.get('date'), errors='coerce')

TARGET_YEAR = int(df_grp['date'].max())
DOMAIN_TO_PLOT = 'social'
domain_df = df_grp.loc[df_grp['domain'].astype(str) == DOMAIN_TO_PLOT].copy()
df_soc_plot = domain_df.loc[domain_df['date'] == TARGET_YEAR].copy()
df_soc_plot['percent'] = pd.to_numeric(df_soc_plot['ratio'], errors='coerce') * 100

# Group definitions (necessary for plotting)
groups_present = list(domain_df['group'].astype(str).unique())
preferred = ['bottom40','middle40','top20','bottom-40','middle-40','top-20']
order_for_plot = [g for g in preferred if g in groups_present] + [g for g in groups_present if g not in preferred]

color_map = { 'bottom40': "#ffd700", 'middle40': "#d8bfd8", 'top20': "#4682b4" }
canonical_colors = { _norm(k): v for k, v in color_map.items() }
palette_dict = { g: canonical_colors.get(_norm(g), '#777777') for g in groups_present }

# Translation map (Code -> Description)
translate_map_up = {
    "CO2":"Lack of internet","CO1":"Lack of public transport","ED1":"Illiteracy rate","ED2":"Incomplete secondary school",
    "EN1":"Lack of electricity","EN2":"Lack of clean fuels indoors","EQ1":"Gender inequality","NU1":"Undernourished",
    "NU2":"Food insecurity","HE1":"Under-5 mortality","HE2":"Lack of health services",
    "HO1":"Slums or informal housing","IW1":"Society poverty","IW2":"Youth NEET","PJ1":"Perceptions of corruption",
    "PJ2":"Homicide rate","PV1":"Autocratic regimes","SC1":"Lack of social support","SC2":"Income inequality",
    "WA1":"Unsafe drinking Water","WA2":"Unsafe sanitation",
}

def translate_indicator(name):
    key = name.strip()
    if key.upper() in translate_map_up:
        return translate_map_up[key.upper()]
    low = key.lower()
    for code, desc in translate_map_up.items():
        if code.lower() in low:
            return desc
    return key.replace('_', ' ').title()

# Sorting
raw_inds = list(df_soc_plot['indCode'].astype(str).unique())
label_for = {k.upper(): v for k, v in translate_map_up.items()}
indicators = sorted(raw_inds, key=lambda k: label_for.get(k.upper(), k).lower())


# --- PLOTTING ---
g = sns.catplot(
    data=df_soc_plot, x='group', y='percent',
    col='indCode', col_order=indicators, kind='bar', col_wrap=3,
    hue='group', hue_order=order_for_plot, palette=palette_dict, order=order_for_plot,
    height=3.0, aspect=0.95, sharey=True, dodge=False, legend=False,
)
g.set_axis_labels("", "") 

fixed_ticks = [0,20,40,60,80,100]
col_wrap = 3
num_indicators = len(indicators)
num_rows = int(np.ceil(num_indicators / col_wrap))

# X-axis labels for the bottom row
group_labels_for_ticks = ['Bottom-40', 'Middle-40', 'Top-20'] 

for i, ax in enumerate(g.axes.flatten()):
    # Y-axis Inversion
    ax.set_ylim(100, 0)
    ax.set_yticks(fixed_ticks)
    ax.set_yticklabels([f"{t}%" for t in fixed_ticks], fontsize=8)

    # 1. Προσθήκη Y-axis Title δίπλα στον άξονα
    if i % col_wrap == 0:
        ax.set_ylabel("Social Shortfall", fontsize=10)
    else:
        ax.set_ylabel("")
        
    # Panel Title (Translation)
    ind_raw = ax.get_title().split('=')[-1].strip()
    ax.set_title(translate_indicator(ind_raw), fontsize=10, y=0.98)
    ax.set_xlabel('')

    is_bottom_row = (i >= num_indicators - col_wrap)

    if is_bottom_row:
        # Εμφάνιση των B40/M40/T20
        # Χρησιμοποιούμε rotation=45 και ha='right' για σωστή γωνία
        ax.set_xticklabels(group_labels_for_ticks, rotation=45, ha='right', fontsize=9)
        # Κρατάμε το tick_params, αλλά αφαιρούμε το rotation=0
        ax.tick_params(axis='x', labelsize=8) 
    else:
        ax.set_xticklabels([])
        ax.tick_params(labelbottom=False)
# g.figure.supylabel("Social Shortfall (%)", fontsize=14, x=0.015) 


g.figure.subplots_adjust(top=0.9, left=0.08, right=0.98, hspace=0.3)
plt.show()

In [ ]:
# --- 0. ΟΡΙΣΜΟΣ MAPS (Indicator Key -> Full Description) ---

map_ecological = {
    'interhemAOD': 'Aerosol optical depth',
    'omega_a': 'Aragonite saturation',
    'blueDev': 'Blue-water flows',
    'co2_ppm': 'CO2 concentration',
    'forestAreaMKM2': 'Forest area',
    'chemicalsMt_Hzd': 'Hazardous chemicals production',
    'hanppGtC': 'Human appropriation of energy produ...',
    'nitrogenMt': 'Nitrogen polution',
    'phosphorusMt': 'Phosphorus polution',
    'erf_wm2': 'Radioactive forcing',
    'soilDev': 'Soil moisture',
    'extinction1900': 'Species extictions',
    'totalOzone': 'Stratospheric ozone concentration',
}

map_social = {
    'govRegimes': 'Autocratic regimes',
    'under5death': 'Child mortality',
    'foodInsecurity': 'Food insecurity',
    'genderGapIndex': 'Gender inequality',
    'homicideOver5': 'Homicides',
    'adultLiteracy': 'Illiteracy',
    'urbanSlums': 'Inadequate housing',
    'palma': 'Income inequality',
    'secondarySchool': 'Incomplete secondary school',
    'energyIndoor': 'Lack of clean fuels indoors',
    'energyAccess': 'Lack of electricity',
    'UHCindex': 'Lack of health services',
    'internet': 'Lack of internet',
    'publicTrans': 'Lack of public transport',
    'socialSupport': 'Lack of social support',
    'controlCorruption': 'Perceptions of corruption',
    'societalPoverty': 'Societal poverty',
    'undernourishment': 'Undernourishment',
    'drinkingH2O': 'Unsafe drinking water',
    'sanitation': 'Unsafe sanitation',
    'youthNEET': 'Youth unemployment',
}
# Βελτιωμένα Χρώματα
BASE_COLOR = "#cd2541"   # Απαλό κόκκινο για τη βάση
BASE_COLOR_SOC = '#a41b23'
WORSE_COLOR = "#ab4c4c" # Σκούρο κόκκινο για επιδείνωση
BETTER_COLOR = "#e65a76" # Απαλό κοραλί/ροζέ για βελτίωση
# --- 1. ΦΟΡΤΩΣΗ ΚΑΙ ΠΡΟΕΤΟΙΜΑΣΙΑ ---
PATH_BAGUETTE = r"a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\7_20250112_baguetteData.csv"

if not os.path.exists(PATH_BAGUETTE):
    raise FileNotFoundError(f"❌ Το αρχείο baguetteData δεν βρέθηκε: {PATH_BAGUETTE}")

df_bag = pd.read_csv(PATH_BAGUETTE)
df_bag.columns = df_bag.columns.str.strip()
df_bag['domain'] = df_bag['domain'].str.strip()

# Ensure valueStart is numeric and drop rows where valueStart is missing (user request #1)
df_bag['valueEnd'] = pd.to_numeric(df_bag['valueEnd'], errors='coerce')
df_bag = df_bag[df_bag['valueEnd'].notna()].reset_index(drop=True)

# compute percents
df_bag['pct_start'] = pd.to_numeric(df_bag.get('ratioStart', 0), errors='coerce') * 100
df_bag['pct_end']   = pd.to_numeric(df_bag.get('ratioEnd',   0), errors='coerce') * 100
def get_bar_segments(df_source, indicator_map):
    df_temp = df_source.copy()
    # Υπολογισμοί
    df_temp['bar_base'] = df_temp[['pct_start', 'pct_end']].min(axis=1)
    df_temp['bar_change'] = (df_temp['pct_end'] - df_temp['pct_start']).abs()
    df_temp['bar_max'] = df_temp[['pct_start', 'pct_end']].max(axis=1)

    # Καθορισμός Χρώματος
    df_temp['color'] = np.where(
        df_temp.get('betterWorse', '') == 'worsening',
        '#ab4c4c',
        '#e65a76'
    )

    # Μετάφραση σε πλήρες όνομα και χρήση του ως κλειδί ταξινόμησης
    df_temp['long_name'] = df_temp['indicator'].map(indicator_map).fillna(df_temp['indicator'])

    # Ταξινόμηση Αλφαβητικά βάσει της πλήρους περιγραφής (long_name)
    return df_temp.sort_values(by='long_name', ascending=True)

df_eco_plot = get_bar_segments(df_bag[df_bag['domain'] == 'ecological'], map_ecological)
df_soc_plot = get_bar_segments(df_bag[df_bag['domain'] == 'social'], map_social)

# --- 3. Δημιουργία Subplots (Sandwich Layout) ---
max_len = max(len(df_eco_plot), len(df_soc_plot))
x_pos_eco = np.arange(len(df_eco_plot))
x_pos_soc = np.arange(len(df_soc_plot))
width = 0.8
# CHANGED: στενότερο (μικρότερο width) και ψηλότερο (μεγαλύτερο height)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.5, 11), gridspec_kw={'height_ratios':[1,1]})
# στενότερα πλαϊνά (αυξάνουμε left, μειώνουμε right) και περισσότερος χώρος πάνω/κάτω
fig.subplots_adjust(left=0.18, right=0.82, top=0.84, bottom=0.06, hspace=0.28)

num_bars = len(df_eco_plot)
num_bars_soc = len(df_soc_plot)

x_pos_eco = np.arange(num_bars)
x_pos_soc = np.arange(num_bars_soc)
labels_eco = df_eco_plot['long_name'].tolist()

df_eco_plot['color'] = np.where(df_eco_plot.get('betterWorse', '') == 'worsening', WORSE_COLOR, BETTER_COLOR)
df_soc_plot['color'] = np.where(df_soc_plot.get('betterWorse', '') == 'worsening', WORSE_COLOR, BETTER_COLOR)


# ----------------------------------------------------
# Plot 1: Ecological Overshoot (UPPER BARS)
# ----------------------------------------------------
ax1.bar(x_pos_eco, df_eco_plot['bar_base'], color=BASE_COLOR, width=width)
ax1.bar(x_pos_eco, df_eco_plot['bar_change'], bottom=df_eco_plot['bar_base'],
        color=df_eco_plot['color'], width=width)

ax1.set_ylim(0, 300)
top_ticks = np.arange(0, 301, 50)
ax1.set_yticks(top_ticks)
ax1.set_yticklabels([f"{t}%" for t in top_ticks], fontsize=8)
ax1.set_ylabel('Ecological Overshoot', fontsize=8)
ax1.axhline(0, color='black', linewidth=0.5)

# X-axis: remove bottom labels, enable top ticks and place equal-spaced labels on top
ax1.set_xticks(x_pos_eco)
ax1.set_xticklabels([''] * num_bars)   # hide bottom labels
ax1.tick_params(axis='x', which='both', bottom=False, top=True,
                labelbottom=False, labeltop=True, labelsize=12)

# set the top labels (rotated so they fit)
ax1.set_xticklabels(labels_eco, rotation=70, ha='left', fontsize=12)



# X-limits to keep spacing stable
ax1.set_xlim(-width/2, num_bars - width/2)
# ----------------------------------------------------
# Plot 2: Social Shortfall (LOWER BARS)
# ----------------------------------------------------
ax2.bar(x_pos_soc, -df_soc_plot['bar_base'], color=BASE_COLOR_SOC, width=width)
ax2.bar(x_pos_soc, -df_soc_plot['bar_change'], bottom=-df_soc_plot['bar_base'],
        color=df_soc_plot['color'], width=width)

ax2.set_ylim(-100, 0)

bot_ticks = np.arange(0, 101, 10)
ax2.set_yticks(-bot_ticks)
ax2.set_yticklabels([f"{t}%" for t in bot_ticks], fontsize=8)
ax2.set_ylabel('Social Shortfall', fontsize=8)
ax2.axhline(0, color='black', linewidth=0.5)

ax2.set_xticks(x_pos_soc)
ax2.set_xticklabels(df_soc_plot['long_name'], rotation=45, ha='right', fontsize=12)

# Στοίχιση/Εμφάνιση
if len(x_pos_eco) and len(x_pos_soc):
    ax1.set_xlim(min(x_pos_eco.min(), x_pos_soc.min()), max(x_pos_eco.max(), x_pos_soc.max()))

plt.tight_layout(rect=[0, 0.03, 1, 0.94])  # αν θέλεις να αλλάξεις το μέγεθος της φιγούρας

plt.show()

In [ ]:
%matplotlib widget
def plot_doughnut_area_scaled(df_dn=None, file_path=None,
                              inner_domain='social', outer_domain='ecological',
                              R_soc=1.0, R_eco=1.5, scale_eco=0.008,
                              gap_frac=0.86, figsize=(10,10), title=None,
                              save_path=None, zoom=1.0,
                              map_social=None, map_ecological=None):
    # load if needed
    if df_dn is None:
        if not file_path or not os.path.exists(file_path):
            raise FileNotFoundError("Provide df_dn or valid file_path")
        df_dn = pd.read_csv(file_path)
        df_dn.columns = df_dn.columns.str.strip()
        df_dn['domain'] = df_dn.get('domain', '').astype(str).str.strip()

    # ensure date numeric
    df_dn['date'] = pd.to_numeric(df_dn.get('date'), errors='coerce')

    # attempt to use YEAR=2022 snapshot for domain-specific values
    YEAR = 2022
    df_latest = df_dn[df_dn['date'] == YEAR].copy()
    use_year_snapshot = False
    if not df_latest.empty:
        # pick rows for each domain at YEAR
        df_soc_latest = df_latest[df_latest['domain'].str.lower() == inner_domain.lower()].copy().reset_index(drop=True)
        df_eco_latest = df_latest[df_latest['domain'].str.lower() == outer_domain.lower()].copy().reset_index(drop=True)
        # require at least some rows for both domains to use snapshot
        if (not df_soc_latest.empty) and (not df_eco_latest.empty):
            use_year_snapshot = True
            # set ratio_pct: social uses 'value' (assumed percent), ecological uses 'shortfallOvershoot_pct'
            if 'value' in df_soc_latest.columns:
                df_soc_latest['ratio_pct'] = pd.to_numeric(df_soc_latest['value'], errors='coerce')
            else:
                df_soc_latest['ratio_pct'] = 0.0
            if 'shortfallOvershoot_pct' in df_eco_latest.columns:
                df_eco_latest['ratio_pct'] = pd.to_numeric(df_eco_latest['shortfallOvershoot_pct'], errors='coerce')
            else:
                df_eco_latest['ratio_pct'] = 0.0

    # fallback: existing ratio-detection logic (if not using snapshot)
    ratio_cols = [c for c in df_dn.columns if 'ratio' in c.lower()]
    prefer = [c for c in ratio_cols if 'end' in c.lower()] + ratio_cols
    raw = prefer[0] if prefer else ('value' if 'value' in df_dn.columns else None)
    if raw:
        df_dn[raw] = pd.to_numeric(df_dn[raw], errors='coerce').fillna(0)
    else:
        df_dn['__ratio_tmp'] = 0.0
        raw = '__ratio_tmp'

    vals = df_dn[raw].astype(float).values
    if vals.size == 0 and raw != '__ratio_tmp':
        # not fatal if we will use the YEAR snapshot; only raise if no data at all and no snapshot
        if not use_year_snapshot:
            raise RuntimeError("No numeric values found for ratio/value")

    # Normalize to percent columns for fallback case
    if vals.size and vals.max() > 1.0:
        df_dn['ratio_pct'] = vals
        df_dn['ratio_scaled'] = (df_dn['ratio_pct'] / 100.0).clip(lower=0)
    else:
        df_dn['ratio_scaled'] = vals.clip(0, 1) if vals.size else np.array([])
        if vals.size:
            df_dn['ratio_pct'] = df_dn['ratio_scaled'] * 100.0
        else:
            df_dn['ratio_pct'] = df_dn.get('ratio_pct', pd.Series(dtype=float))

    # split domains — prefer YEAR snapshot if available
    if use_year_snapshot:
        df_soc = df_soc_latest.copy().reset_index(drop=True)
        df_eco = df_eco_latest.copy().reset_index(drop=True)
        # ensure no NaNs in ratio_pct
        df_soc['ratio_pct'] = pd.to_numeric(df_soc['ratio_pct'], errors='coerce').fillna(0.0)
        df_eco['ratio_pct'] = pd.to_numeric(df_eco['ratio_pct'], errors='coerce').fillna(0.0)
    else:
        df_soc = df_dn[df_dn['domain'].str.lower() == inner_domain.lower()].copy().reset_index(drop=True)
        df_eco = df_dn[df_dn['domain'].str.lower() == outer_domain.lower()].copy().reset_index(drop=True)
        # ensure ratio_pct exists
        if 'ratio_pct' not in df_soc.columns:
            df_soc['ratio_pct'] = pd.to_numeric(df_soc.get(raw, 0), errors='coerce').fillna(0.0)
        if 'ratio_pct' not in df_eco.columns:
            df_eco['ratio_pct'] = pd.to_numeric(df_eco.get(raw, 0), errors='coerce').fillna(0.0)

    if df_soc.empty or df_eco.empty:
        raise RuntimeError("Inner or outer domain empty — check CSV / 'domain' column / YEAR snapshot")

    # <<< ADDED: αριθμός στοιχείων και γωνίες για τις ράβδους >>>
    N_soc = len(df_soc)
    N_eco = len(df_eco)

    # προστασία για περίπτωση 0 (αποφεύγουμε διαίρεση με το 0)
    theta_soc = np.linspace(np.pi, -np.pi, N_soc, endpoint=False) if N_soc > 0 else np.array([])
    theta_eco = np.linspace(np.pi, -np.pi, N_eco, endpoint=False) if N_eco > 0 else np.array([])

    width_soc = (2 * np.pi / max(1, N_soc)) * gap_frac
    width_eco = (2 * np.pi / max(1, N_eco)) * gap_frac

    # radii and heights
    s_pct = df_soc['ratio_pct'].clip(0, 100).values
    r_inner = R_soc * np.sqrt(np.maximum(0.0, 1.0 - (s_pct / 100.0)))
    height_soc = R_soc - r_inner

    o_pct = df_eco['ratio_pct'].clip(lower=0).values
    r_outer = R_eco * np.sqrt(1.0 + (o_pct * scale_eco))
    height_eco = r_outer - R_eco

    # colour helpers
    def to_mpl_colors(col_series, n):
        if col_series is None: return None
        try:
            arr = np.array(col_series.tolist() if hasattr(col_series, "tolist") else col_series)
            if len(arr) == int(n) and all(isinstance(x, str) and x.strip().startswith('#') for x in arr):
                return arr.tolist()
        except Exception:
            pass
        return None

    def build_scaled_colors(values, light_hex, dark_hex):
        vals = np.array(values, dtype=float)
        if vals.size == 0: return []
        finite = vals[np.isfinite(vals)]
        if finite.size == 0: return []
        pos = finite[finite > 0]
        if pos.size > 0:
            vmin_f = float(np.nanmin(pos))
            vmax_f = float(np.nanmax(pos))
        else:
            vmin_f, vmax_f = 0.0, float(np.nanmax(finite))
        if np.isclose(vmax_f, vmin_f):
            t = np.zeros_like(vals, dtype=float)
            t[(vals > 0) & np.isfinite(vals)] = 1.0
        else:
            t_raw = (vals - vmin_f) / (vmax_f - vmin_f)
            t = np.clip(np.where(np.isfinite(t_raw), t_raw, 0.0), 0.0, 1.0)
            t = np.where(vals <= 0, 0.0, t)
        light_rgb = np.array(mcolors.to_rgb(light_hex))
        dark_rgb = np.array(mcolors.to_rgb(dark_hex))
        rgbs = (1.0 - t[:, None]) * light_rgb + t[:, None] * dark_rgb
        return [mcolors.to_hex(rgb) for rgb in rgbs]

    inner_from_col = to_mpl_colors(df_soc.get('color'), N_soc)
    outer_from_col = to_mpl_colors(df_eco.get('color'), N_eco)
    light_hex = '#ffc6c4'
    dark_hex = '#672044'

    if inner_from_col is not None:
        inner_colors = inner_from_col
    else:
        inner_colors = build_scaled_colors(df_soc['ratio_pct'].values, light_hex, dark_hex)

    if outer_from_col is not None:
        outer_colors = outer_from_col
    else:
        outer_colors = build_scaled_colors(df_eco['ratio_pct'].values, light_hex, dark_hex)

    # prepare lookup maps
    def _norm_key(k): return str(k).strip().lower()
    label_map_soc_norm = { _norm_key(k): v for k, v in map_social.items() }
    label_map_eco_norm = { _norm_key(k): v for k, v in map_ecological.items() }

    soc_keys_raw = df_soc.get('indCode', df_soc.get('indicator', pd.Series([str(i) for i in range(N_soc)]))).astype(str).values
    eco_keys_raw = df_eco.get('indCode', df_eco.get('indicator', pd.Series([str(i) for i in range(N_eco)]))).astype(str).values
    soc_long = df_soc.get('dimension') if 'dimension' in df_soc.columns else None
    eco_long = df_eco.get('dimension') if 'dimension' in df_eco.columns else None

    # figure + polar axis
    fig = plt.figure(figsize=figsize, dpi=100)
    ax = fig.add_subplot(1, 1, 1, projection='polar')
    ax.set_facecolor('white')

    xs = np.linspace(0, 2*np.pi, 400)
    total_band = float(max(1e-6, R_eco - R_soc))
    prop_inner, prop_middle, prop_outer = 0.15, 0.70, 0.15
    inner_thick = total_band * prop_inner
    middle_thick = total_band * prop_middle
    outer_thick = total_band * prop_outer

    inner_r0 = R_soc
    inner_r1 = inner_r0 + inner_thick
    middle_r0 = inner_r1
    middle_r1 = middle_r0 + middle_thick
    outer_r0 = middle_r1
    outer_r1 = R_eco

    dark_green = "#227443"
    mid_green = "#6eb446"

    ax.fill_between(xs, inner_r0, inner_r1, color=dark_green, alpha=1.0, zorder=0, linewidth=0)
    ax.fill_between(xs, middle_r0, middle_r1, color=mid_green, alpha=1.0, zorder=1, linewidth=0)
    ax.fill_between(xs, outer_r0, outer_r1, color=dark_green, alpha=1.0, zorder=2, linewidth=0)
    ax.fill_between(xs, R_soc, R_eco, color='#6eb446', alpha=0.28, zorder=0, linewidth=0)

    # build per-item label list for inner (use provided map or df)
    if 'dimension' in df_soc.columns:
        raw_dims = df_soc['dimension'].astype(str).tolist()
    else:
        raw_dims = []
        for i, key in enumerate(soc_keys_raw):
            d = label_map_soc_norm.get(_norm_key(key)) or (soc_long.iloc[i] if soc_long is not None else str(key))
            raw_dims.append(str(d))

    label_to_indices = OrderedDict()
    for idx, lab in enumerate(raw_dims):
        lab = lab.strip()
        label_to_indices.setdefault(lab, []).append(idx)

    # faint wedges per distinct label
    nsteps = 48
    for lab, idxs in label_to_indices.items():
        ang_starts = [float(theta_soc[i] - width_soc/2.0) for i in idxs]
        ang_ends   = [float(theta_soc[i] + width_soc/2.0) for i in idxs]
        ang0 = min(ang_starts); ang1 = max(ang_ends)
        a0 = ang0 % (2*np.pi); a1 = ang1 % (2*np.pi)
        if a1 < a0:
            th1 = np.linspace(a0, 2*np.pi, nsteps//2, endpoint=False)
            th2 = np.linspace(0.0, a1, nsteps - th1.size)
            th = np.concatenate([th1, th2])
        else:
            th = np.linspace(a0, a1, nsteps)
        r0_max = float(np.max(r_inner[np.array(idxs, dtype=int)]))
        try:
            wedge_col = inner_colors[idxs[0]]
        except Exception:
            wedge_col = inner_colors if isinstance(inner_colors, str) else '#ffffff'
        ax.fill_between(th, 0.0, r0_max, color=wedge_col, alpha=0.12, zorder=2, linewidth=0)

    # draw bars
    inner_bars = ax.bar(theta_soc, height_soc, width=width_soc, bottom=r_inner,
                        color=inner_colors, edgecolor=None, linewidth=0, zorder=3, align='center', antialiased=False)
    outer_bars = ax.bar(theta_eco, height_eco, width=width_eco, bottom=R_eco,
                        color=outer_colors, edgecolor=None, linewidth=0, zorder=4, align='center', antialiased=False)

    # text sizing defaults (base sizes)
    title_fs = 14
    outer_arc_fs = 10
    inner_arc_fs = 10
    center_label_base_fs = 8
    eco_label_base_fs = 8
    rescale_base_fs = 12.0

    # per-character curved titles
    def _char_arc_draw(ax, text, radius, theta_center=np.pi/2.0, arc_span=np.pi*0.8,
                        fontsize=14, color='white', zorder=7):
        txt = str(text or "")
        if not txt: return []
        chars = list(txt)
        n = len(chars)
        angles = theta_center + np.linspace(arc_span/2.0, -arc_span/2.0, n)
        handles = []
        for ch, ang in zip(chars, angles):
            if ch == ' ': continue
            rot = np.rad2deg(ang) - 90.0
            if rot > 90: rot -= 180
            if rot < -90: rot += 180
            h = ax.text(ang, radius, ch,
                        rotation=rot, rotation_mode='anchor',
                        ha='center', va='center',
                        fontsize=fontsize, fontweight='bold',
                        color=color, zorder=zorder, clip_on=False)
            handles.append(h)
        return handles

    outer_mid_r = outer_r0 + (outer_r1 - outer_r0) * 0.5
    inner_mid_r = inner_r0 + (inner_r1 - inner_r0) * 0.5

    arc_texts_outer = _char_arc_draw(ax, "ECOLOGICAL CEILING", outer_mid_r,
                                     theta_center=np.pi/2.0, arc_span=np.pi*0.52,
                                     fontsize=outer_arc_fs, color='white', zorder=7)
    arc_texts_inner = _char_arc_draw(ax, "SOCIAL FOUNDATION", inner_mid_r,
                                     theta_center=np.pi/2.0, arc_span=np.pi*0.44,
                                     fontsize=inner_arc_fs, color='white', zorder=7)

    # helper for annotation rotation/alignment
    def _text_props(mid_angle):
        deg = np.rad2deg(mid_angle)
        rot = deg - 90.0
        if rot > 90: rot -= 180
        if rot < -90: rot += 180
        ha = 'left' if np.cos(mid_angle) >= 0 else 'right'
        return rot, ha

    def two_line_label(s):
        s = str(s).strip()
        parts = s.split()
        if len(parts) <= 1: return s
        if len(parts) == 2: return parts[0] + '\n' + parts[1]
        k = len(parts) // 2
        return ' '.join(parts[:k]) + '\n' + ' '.join(parts[k:])

    # middle ring centers
    middle_center_r = inner_r1 + (middle_r1 - middle_r0) * 0.5
    centers = []
    for lab, idxs in label_to_indices.items():
        ang_pts = []
        for i in idxs:
            ang_pts.append(float(theta_soc[i] - width_soc/2.0))
            ang_pts.append(float(theta_soc[i] + width_soc/2.0))
        ang_pts = np.array(ang_pts)
        mean_ang = np.arctan2(np.mean(np.sin(ang_pts)), np.mean(np.cos(ang_pts)))
        if mean_ang < 0: mean_ang += 2*np.pi
        centers.append((lab, mean_ang))

    center_texts = []
    for lab, mid in centers:
        wrapped = two_line_label(lab)
        ann = ax.annotate(
            wrapped,
            xy=(mid, middle_center_r),
            xytext=(0, 0),
            textcoords='offset points',
            xycoords='data',
            ha='center', va='center',
            fontsize=center_label_base_fs, color='white',
            zorder=7,
            rotation=_text_props(mid)[0],
            rotation_mode='anchor',
            bbox=dict(boxstyle='square,pad=0.1', fc='none', ec='none'),
            clip_on=False
        )
        center_texts.append(ann)

    # ecological labels outside outer ring
    eco_label_offset_frac = 0.18
    extra_pad = 0.02   # επιπλέον σταθερό padding
    eco_label_r = outer_r1 + (total_band * eco_label_offset_frac) + extra_pad
    eco_label_texts = []
    eco_texts_raw = []
    for i, key in enumerate(eco_keys_raw):
        lbl = label_map_eco_norm.get(_norm_key(key))
        if not lbl and eco_long is not None:
            try: lbl = str(eco_long.iloc[i])
            except Exception: lbl = None
        if not lbl: lbl = str(key)
        eco_texts_raw.append(lbl.strip())

    label_to_indices_eco = OrderedDict()
    for idx, lab in enumerate(eco_texts_raw):
        label_to_indices_eco.setdefault(lab, []).append(idx)

    for lab, idxs in label_to_indices_eco.items():
        ang_pts = []
        for i in idxs:
            ang_pts.append(float(theta_eco[i] - width_eco/2.0))
            ang_pts.append(float(theta_eco[i] + width_eco/2.0))
        ang_pts = np.array(ang_pts)
        mean_ang = np.arctan2(np.mean(np.sin(ang_pts)), np.mean(np.cos(ang_pts)))
        if mean_ang < 0: mean_ang += 2*np.pi
        rot, ha = _text_props(mean_ang)
        wrapped = two_line_label(lab)
        ann = ax.annotate(
            wrapped,
            xy=(mean_ang, eco_label_r),
            xycoords='data',
            textcoords='data',
            ha=ha, va='center',
            fontsize=eco_label_base_fs, zorder=6, color='black',
            rotation=rot, rotation_mode='anchor',
            bbox=dict(boxstyle='square,pad=0.1', fc='none', ec='none'),
            clip_on=False
        )
        eco_label_texts.append(ann)

    # compute visible_top & set axis limits BEFORE connecting rescale
    try:
        max_outer = float(np.max(r_outer)) if 'r_outer' in locals() or 'r_outer' in globals() else R_eco
    except Exception:
        max_outer = R_eco
    pad = 0.01
    top_needed = max(max_outer, eco_label_r) + pad

    # ---- Use only the function's zoom parameter (caller decides). Fallback = 1.0 ----
    try:
        zoom_f = float(zoom)
    except Exception:
        zoom_f = 1.0
    visible_top = R_soc + (top_needed - R_soc) * zoom_f
    visible_top = max(visible_top, R_soc + 1e-6)

    ax.set_ylim(0, visible_top)
    ax.set_yticklabels([])
    ax.set_xticks([])
    ax.grid(False)
    if 'polar' in ax.spines:
        ax.spines['polar'].set_visible(False)

    ax.set_title(title, fontsize=title_fs, y=1.06)
    plt.tight_layout(pad=0.6, rect=[0,0,1,1])

    # dynamic rescale handler (uses reference_span computed now)
    reference_span = float(visible_top - ax.get_ylim()[0]) if visible_top is not None else float(R_eco - R_soc)

    def _rescale_texts(event=None):
        try:
            ylim = ax.get_ylim()
            span = float(ylim[1] - ylim[0])
            if span <= 0: return
            fs = rescale_base_fs * (reference_span / span)
            fs = max(6.0, min(36.0, fs))
            # scale annotations proportionally
            scale = fs / rescale_base_fs if rescale_base_fs else 1.0
            # center + eco annotations
            for t in (center_texts + eco_label_texts):
                try:
                    t.set_fontsize(max(6.0, center_label_base_fs * scale))
                except Exception:
                    pass
            # arc texts
            for h in arc_texts_outer:
                try:
                    h.set_fontsize(max(4.0, outer_arc_fs * scale))
                except Exception:
                    pass
            for h in arc_texts_inner:
                try:
                    h.set_fontsize(max(4.0, inner_arc_fs * scale))
                except Exception:
                    pass
            fig.canvas.draw_idle()
        except Exception:
            pass

    try:
        fig.canvas.mpl_connect('draw_event', _rescale_texts)
    except Exception:
        pass

    # optional tooltips (mplcursors)
    try:
        import mplcursors
    except Exception:
        mplcursors = None

    if mplcursors is not None:
        patches = []
        if inner_bars is not None:
            patches.extend(list(inner_bars))
        if outer_bars is not None:
            patches.extend(list(outer_bars))
        if patches:
            cursor = mplcursors.cursor(patches, hover=True)
            @cursor.connect("add")
            def on_add(sel):
                artist = sel.artist
                try:
                    idx = patches.index(artist)
                except ValueError:
                    return
                n_inner = len(list(inner_bars)) if inner_bars is not None else 0
                if idx < n_inner:
                    i = idx
                    key = soc_keys_raw[i]
                    pct = float(df_soc['ratio_pct'].iloc[i])
                    desc = label_map_soc_norm.get(_norm_key(key), str(key))
                    ax_local = inner_bars[0].axes if len(inner_bars) else artist.axes
                else:
                    i = idx - n_inner
                    key = eco_keys_raw[i]
                    pct = float(df_eco['ratio_pct'].iloc[i])
                    desc = label_map_eco_norm.get(_norm_key(key), str(key))
                    ax_local = outer_bars[0].axes if len(outer_bars) else artist.axes
                sel.annotation.set_text(f"{desc}: {pct:.2f}%")
                try:
                    sel.annotation.set_transform(ax_local.transAxes)
                    sel.annotation.xy = (0.5, 0.98)
                    sel.annotation.set_position((0.5, 0.98))
                    sel.annotation.set_ha('center'); sel.annotation.set_va('top')
                except Exception:
                    pass
                try:
                    sel.annotation.set_draggable(False)
                except Exception:
                    pass

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')

    return fig, ax

file_path_global_data = r"a-fanning-doughnut-v3-a0460e5\Analysis-Final\myData\12_20250515_globalDoughnutData_2000-2022.csv"
MAP_SOCIAL_EXAMPLE = {'S1':'Adequate Housing','S2':'Access to Energy'}
MAP_ECOLOGICAL_EXAMPLE = {'E1':'Climate Change','E2':'Ocean Acidification'}
fig, ax = plot_doughnut_area_scaled(file_path=file_path_global_data, figsize=(12,12), zoom=0.6,
                                    map_social=MAP_SOCIAL_EXAMPLE, map_ecological=MAP_ECOLOGICAL_EXAMPLE)
plt.show()